# jaxoom T4 compiler introspection

This notebook runs only the 14-case compiler diagnostic manifest. It pins JAX 0.11.0, checks for a Tesla T4, checks out one immutable jaxoom commit, captures compact compiler summaries, inventories targeted XLA dumps, and packages the evidence.

No calibration constants or estimator code are changed.


In [5]:
# Install before importing JAX. The CUDA plugin is supplied by the JAX extra.
import os
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]==0.11.0"], check=True)
print("Pinned JAX installation completed")


Pinned JAX installation completed


In [6]:
from pathlib import Path
import shutil
import subprocess

REPO_URL = "https://github.com/Slavov88/jaxoom.git"
PINNED_COMMIT = "2ed40dd81536f132b50fba311155dab0cb27b8d2"
REPO = Path("/content/jaxoom_compiler_introspection")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", PINNED_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == PINNED_COMMIT, (actual_commit, PINNED_COMMIT)
MANIFEST = REPO / "experiments" / "compiler_drift_cases_2026-09-08.json"
HARNESS = REPO / "experiments" / "compiler_introspection.py"
assert MANIFEST.exists() and HARNESS.exists()
print({"repository": str(REPO), "commit": actual_commit})


{'repository': '/content/jaxoom_compiler_introspection', 'commit': '2ed40dd81536f132b50fba311155dab0cb27b8d2'}


In [7]:
# Import JAX only after the pinned installation and repository checkout.
import json
import platform
import re
import time
import jax

assert jax.__version__ == "0.11.0", jax.__version__
assert jax.default_backend() == "gpu", jax.default_backend()
devices = jax.devices()
assert devices, "No accelerator detected"
device_kinds = [getattr(device, "device_kind", "") for device in devices]
assert any("T4" in kind.upper() for kind in device_kinds), device_kinds
print({"jax": jax.__version__, "backend": jax.default_backend(), "devices": [str(d) for d in devices], "device_kinds": device_kinds})


{'jax': '0.11.0', 'backend': 'gpu', 'devices': ['cuda:0'], 'device_kinds': ['Tesla T4']}


In [8]:
# Run the case-ID-driven harness in a fresh subprocess with targeted XLA dumps.
import os
import subprocess

ROOT = Path("/content/jaxoom_t4_compiler_introspection_output")
if ROOT.exists():
    shutil.rmtree(ROOT)
DUMPS = ROOT / "compiler_dumps"
RESULT = ROOT / "compiler_introspection_t4.json"
LOG = ROOT / "execution.log"
ROOT.mkdir(parents=True)
env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO / 'src'}:{REPO / 'experiments'}"
env["JAX_PLATFORMS"] = "cuda"
command = [sys.executable, str(HARNESS), "--manifest", str(MANIFEST), "--output", str(RESULT), "--dump-dir", str(DUMPS), "--xla-flags=--xla_gpu_autotune_level=0", "--expected-device", "none"]
completed = subprocess.run(command, env=env, text=True, capture_output=True)
newline = chr(10)
LOG.write_text("$ " + " ".join(command) + newline + newline + "STDOUT" + newline + completed.stdout + newline + "STDERR" + newline + completed.stderr, encoding="utf-8")
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(f"introspection harness failed with exit code {completed.returncode}")


{
  "status": "COMPUTATIONALLY VERIFIED",
  "case_count": 14,
  "failed": 0,
  "output": "/content/jaxoom_t4_compiler_introspection_output/compiler_introspection_t4.json"
}



In [9]:
# Validate the compact evidence before packaging. Partial case failures remain visible in the ZIP.
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
result = json.loads(RESULT.read_text(encoding="utf-8"))
expected_ids = [row["case_id"] for row in manifest["cases"]]
actual_ids = [row["case_id"] for row in result["cases"]]
assert len(expected_ids) == len(set(expected_ids))
assert actual_ids == expected_ids, (actual_ids, expected_ids)
assert len(actual_ids) == 14
assert result["capture_status"] == "COMPLETE"
assert result["reproduction_status"] == "NOT_REQUESTED"
for row in result["cases"]:
    assert row["status"] == "OK"
    assert "memory_analysis" in row
    memory = row["memory_analysis"]
    assert memory["accounted_bytes"] == (memory["argument_size_in_bytes"] + memory["output_size_in_bytes"] + memory["temp_size_in_bytes"] - memory["alias_size_in_bytes"])
    assert row["dump_delta"]["file_count"] > 0
    assert row["dump_delta"]["memory_usage_reports"]
metadata = {
    "status": result["status"],
    "capture_status": result.get("capture_status"),
    "reproduction_status": result.get("reproduction_status"),
    "xla_flags": result["environment"].get("xla_flags"),
    "repository_commit": actual_commit,
    "jax_version": jax.__version__,
    "backend": jax.default_backend(),
    "device_kinds": device_kinds,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "nvidia_smi": subprocess.run(["nvidia-smi"], text=True, capture_output=True).stdout,
    "case_ids": actual_ids,
    "successful_cases": sum(row["status"] == "OK" for row in result["cases"]),
}
(ROOT / "t4_environment.json").write_text(json.dumps(metadata, indent=2, sort_keys=True) + chr(10), encoding="utf-8")
(ROOT / MANIFEST.name).write_text(MANIFEST.read_text(encoding="utf-8"), encoding="utf-8")
print(json.dumps(metadata, indent=2, sort_keys=True))


{
  "backend": "gpu",
  "capture_status": "COMPLETE",
  "case_ids": [
    "matmul-1024x4096x1024-float32",
    "mlp-b256-w2048-d2-float32",
    "attention-b1-h8-s512-d64-float32",
    "convolution-128x128x16x32-float32",
    "convolution-256x256x32x64-float32",
    "autodiff-b128-w256-float32",
    "autodiff-b256-w512-float32",
    "training-b512-w1024-float32",
    "mlp-b256-w2048-d2-float16",
    "attention-b1-h8-s512-d64-float16",
    "convolution-128x128x16x32-float16",
    "convolution-256x256x32x64-float16",
    "autodiff-b128-w256-float16",
    "autodiff-b256-w512-float16"
  ],
  "device_kinds": [
    "Tesla T4"
  ],
  "jax_version": "0.11.0",
  "nvidia_smi": "Tue Sep  8 10:37:51 2026       \n+-----------------------------------------------------------------------------------------+\n| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |\n+-----------------------------------------+------------------------+----------------------+\n| GPU  Name 

In [10]:
# Package compact evidence and selected raw compiler dumps only.
import zipfile

ZIP = Path("/content/jaxoom_t4_compiler_introspection.zip")
if ZIP.exists():
    ZIP.unlink()
with zipfile.ZipFile(ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (RESULT, ROOT / "t4_environment.json", ROOT / MANIFEST.name, LOG):
        archive.write(path, path.relative_to(ROOT))
    if DUMPS.exists():
        for path in DUMPS.rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(ROOT))
print({"zip": str(ZIP), "size_bytes": ZIP.stat().st_size})


{'zip': '/content/jaxoom_t4_compiler_introspection.zip', 'size_bytes': 537144}


In [11]:
# The download cell is intentionally last.
from google.colab import files
print(f"Generated: {ZIP}")
files.download(str(ZIP))


Generated: /content/jaxoom_t4_compiler_introspection.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>